# Week 4 — Final Evaluation

1,000-episode simulation harness for evaluating all pricing agents 
(heuristics, Q-Learning, DQN) across full booking seasons.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from pricing_env import PricingEnv
from baseline_agents import FixedPriceAgent, TimeBasedDiscountAgent, DemandBasedAgent


def run_large_scale_evaluation(agent, env, n_episodes=1000, has_reset=False):
    """
    Runs any agent (heuristic, Q-Learning, or DQN) across n_episodes 
    full booking seasons and returns detailed per-episode statistics.
    """
    episode_revenues = []
    episode_sell_through = []

    for ep in range(n_episodes):
        obs, info = env.reset()
        if has_reset:
            agent.reset()
        total_reward = 0
        initial_inventory = env.max_inventory

        done = False
        while not done:
            action = agent.act(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated

        episode_revenues.append(total_reward)
        remaining_inventory = obs[0]
        sell_through = (initial_inventory - remaining_inventory) / initial_inventory
        episode_sell_through.append(sell_through)

    return {
        "revenues": episode_revenues,
        "sell_through_rates": episode_sell_through,
        "mean_revenue": np.mean(episode_revenues),
        "std_revenue": np.std(episode_revenues),
        "mean_sell_through": np.mean(episode_sell_through)
    }

In [ ]:
env = PricingEnv()
test_agent = FixedPriceAgent()

results = run_large_scale_evaluation(test_agent, env, n_episodes=1000)
print(f"Mean Revenue: {results['mean_revenue']:.2f}")
print(f"Std Dev: {results['std_revenue']:.2f}")
print(f"Sell-Through Rate: {results['mean_sell_through']*100:.1f}%")

## DQN Multi-Season Price Trajectories

In [ ]:
import torch
from dqn_agent import DQNAgent
from plotting_utils import plot_multi_season_trajectories

dqn_agent = DQNAgent()
dqn_agent.policy_net.load_state_dict(torch.load('../outputs/dqn_checkpoints/dqn_ep2000.pt'))
dqn_agent.policy_net.eval()
dqn_agent.epsilon = 0.0  # greedy for evaluation

plot_multi_season_trajectories(
    dqn_agent, env, n_seasons=5, agent_type="dqn",
    title="Trained DQN — Price Trajectory Across 5 Sample Seasons",
    save_path='../outputs/dqn_multi_season_trajectories.png'
)